# Bisection check (Rust)

Start JupyterLab with `workshop run -- notebook`. Uses the `thole` facade crate from the working tree. After changing the Rust code, run the `:dep` cell again.

In [2]:
:dep thole = { path = "/project/crates/thole" }

In [3]:
use thole::roots::{bisect, Error, Options};

## 1. A first call

The Python binding runs this same code, so the bits must equal the `hex` printed by the Python notebook.

In [4]:
let r = bisect(|x| x * x - 2.0, 0.0, 2.0, &Options::default()).unwrap();
println!("{r:?}");
println!("error = {:e}", (r.x - 2.0_f64.sqrt()).abs());
println!("bits  = {:#018x}", r.x.to_bits());

Root { x: 1.4142135623715149, lo: 1.4142135623696959, hi: 1.4142135623733338, iterations: 39 }
error = 1.5802914532514478e-12
bits  = 0x3ff6a09e667f2000


## 2. Reference roots (mpmath, 50 digits)

In [5]:
let opts = Options::default();
let cases: [(&str, fn(f64) -> f64, f64, f64, f64); 5] = [
    ("x^3 - x - 2", |x| x * x * x - x - 2.0, 1.0, 2.0, 1.5213797068045676),
    ("cos x - x", |x| x.cos() - x, 0.0, 1.0, 0.7390851332151607),
    ("x e^x - 1", |x| x * x.exp() - 1.0, 0.0, 1.0, 0.5671432904097838),
    ("x^2 - 2", |x| x * x - 2.0, 0.0, 2.0, std::f64::consts::SQRT_2),
    ("cos x", f64::cos, 0.0, 2.0, std::f64::consts::FRAC_PI_2),
];
for (name, f, a, b, expected) in cases {
    let r = bisect(f, a, b, &opts).unwrap();
    let err = (r.x - expected).abs();
    println!("{name:12} x={:.16} err={err:.3e} steps={} ok={}", r.x, r.iterations, err <= opts.tol.bound(r.x));
}

x^3 - x - 2  x=1.5213797068045096 err=5.795e-14 steps=38 ok=true
cos x - x    x=0.7390851332147577 err=4.030e-13 steps=38 ok=true
x e^x - 1    x=0.5671432904109679 err=1.184e-12 steps=38 ok=true
x^2 - 2      x=1.4142135623715149 err=1.580e-12 steps=39 ok=true
cos x        x=1.5707963267941523 err=7.443e-13 steps=39 ok=true


()

## 3. The step count predicted by the Lean proof

Smallest `n` with `(b - a) / 2^(n + 1) <= xtol`, with the relative tolerance set to zero.

In [6]:
use thole::core::Tolerance;

for (a, b, xtol) in [(0.0, 1.0, 1e-3), (0.0, 1.0, 1e-6), (-3.0, 5.0, 1e-9), (1.0, 2.0, 0.25), (0.0, 1.0, 0.5)] {
    let mut predicted = 0;
    while (b - a) / 2f64.powi(predicted + 1) > xtol {
        predicted += 1;
    }
    let target = a + 0.3 * (b - a);
    let opts = Options { tol: Tolerance::new(xtol, 0.0), max_iter: 100 };
    let r = bisect(|x| x - target, a, b, &opts).unwrap();
    println!("[{a}, {b}] xtol={xtol:e} steps={} predicted={predicted} half width={:.3e}", r.iterations, (r.hi - r.lo) / 2.0);
    assert_eq!(r.iterations, predicted as usize);
}

[0, 1] xtol=1e-3 steps=9 predicted=9 half width=9.766e-4
[0, 1] xtol=1e-6 steps=19 predicted=19 half width=9.537e-7
[-3, 5] xtol=1e-9 steps=32 predicted=32 half width=9.313e-10
[1, 2] xtol=2.5e-1 steps=1 predicted=1 half width=2.500e-1
[0, 1] xtol=5e-1 steps=0 predicted=0 half width=5.000e-1


()

## 4. Failures

In [7]:
let opts = Options::default();
let few = Options { max_iter: 3, ..opts };
let zero = Options { tol: Tolerance::new(0.0, 0.0), max_iter: 100 };
let negative = Options { tol: Tolerance::new(-1.0, 0.0), max_iter: 100 };
let results: Vec<(&str, Result<_, Error>)> = vec![
    ("no sign change", bisect(|x| x * x + 1.0, 0.0, 2.0, &opts)),
    ("a > b", bisect(|x| x, 1.0, -1.0, &opts)),
    ("infinite endpoint", bisect(|x| x, f64::NEG_INFINITY, 1.0, &opts)),
    ("negative tolerance", bisect(|x| x, -1.0, 1.0, &negative)),
    ("f returns NaN", bisect(|_| f64::NAN, 0.0, 1.0, &opts)),
    ("max_iter too small", bisect(f64::cos, 0.0, 2.0, &few)),
    ("tolerance below spacing", bisect(|x| x - 1e10 / 3.0, 0.0, 1e10, &zero)),
];
for (name, r) in &results {
    match r {
        Ok(root) => println!("{name:24} -> Ok({root:?})"),
        Err(e) => println!("{name:24} -> {e}   [{e:?}]"),
    }
}

no sign change           -> no sign change over the bracket: f(a) = 1, f(b) = 5   [NoSignChange { fa: 1.0, fb: 5.0 }]
a > b                    -> invalid bracket [1, -1]: need finite a <= b   [InvalidBracket { a: 1.0, b: -1.0 }]
infinite endpoint        -> invalid bracket [-inf, 1]: need finite a <= b   [InvalidBracket { a: -inf, b: 1.0 }]
negative tolerance       -> invalid tolerance (abs = -1, rel = 0): both must be non-negative   [InvalidTolerance { abs: -1.0, rel: 0.0 }]
f returns NaN            -> the function returned NaN at x = 0   [NanValue { x: 0.0 }]
max_iter too small       -> tolerance not reached after 3 iterations; last bracket [1.5, 1.75]   [MaxIterations { max_iter: 3, lo: 1.5, hi: 1.75 }]
tolerance below spacing  -> tolerance not reached after 100 iterations; last bracket [3333333333.333333, 3333333333.3333335]   [MaxIterations { max_iter: 100, lo: 3333333333.333333, hi: 3333333333.3333335 }]


()

## 5. Edge cases

In [8]:
let edge: [(&str, fn(f64) -> f64, f64, f64); 5] = [
    ("root at a", |x| x, 0.0, 1.0),
    ("root at b", |x| x - 1.0, 0.0, 1.0),
    ("a == b on a root", |x| x - 1.0, 1.0, 1.0),
    ("pole of 1/x", |x| 1.0 / x, -1.0, 2.0),
    ("three roots", |x| x * x * x - x, -2.0, 3.0),
];
for (name, f, a, b) in edge {
    println!("{name:18} {:?}", bisect(f, a, b, &Options::default()));
}

root at a          Ok(Root { x: 1.8189894035458565e-12, lo: 0.0, hi: 3.637978807091713e-12, iterations: 38 })
root at b          Ok(Root { x: 0.999999999998181, lo: 0.999999999996362, hi: 1.0, iterations: 38 })
a == b on a root   Ok(Root { x: 1.0, lo: 1.0, hi: 1.0, iterations: 0 })
pole of 1/x        Ok(Root { x: 4.547473508864641e-13, lo: -9.094947017729282e-13, hi: 1.8189894035458565e-12, iterations: 40 })
three roots        Ok(Root { x: 1.0000000000006821, lo: 0.9999999999995453, hi: 1.000000000001819, iterations: 41 })


()